# Notebook 2 Lab: Filtering, Fourier, and Sampling

This lab explores linear filtering, Fourier transforms, and sampling through interactive experiments. The reusable implementations live in `filtering.py`. You are invited to inspect the code yourself or ask an LLM to explain, install, or modify pieces of the notebook. Your job is to run experiments, change parameters, use your own images, find failure cases, and explain what assumptions were used or violated.

This notebook is a lab, not a tutorial. The algorithms are already implemented. Your task is to build intuition by making things happen on purpose.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from image_utils import load_image, show
from filtering import (
    HIDDEN_HYBRID_TARGET_PARAMS,
    anti_alias_downsample,
    as_float_image,
    bilateral_filter,
    bilateral_edge_widget,
    bilateral_widget,
    blur_comparison_widget,
    checkerboard_pattern,
    coded_exposure_frequency_widget,
    convolve_image,
    convolution_theorem_widget,
    covariance_explorer,
    downsampling_widget,
    fft_image,
    fft_magnitude_phase,
    fft_magnitude_phase_widget,
    frequency_filter,
    frequency_filter_widget,
    fourier_1d_reconstruction_widget,
    fourier_component_widget,
    fourier_radius_reconstruction_widget,
    fourier_phase_swap_demo,
    gaussian_blur,
    gradient_widget,
    hybrid_image_widget,
    image_gradients,
    interpolation_widget,
    kernel_playground,
    make_box_filter,
    make_derivative_filter,
    make_exposure_kernel,
    make_frequency_mask,
    make_gaussian_point_image,
    make_gaussian_filter,
    make_hybrid_image,
    make_sharpening_filter,
    matrix_gaussian_explorer,
    parse_exposure_pattern,
    resize_image,
    rgb_to_gray,
    sample_image,
    sampling_widget,
    sharpening_widget,
    show_exposure_schedule_point_responses,
    show_frequency_responses,
    show_image_case_grid,
    show_regularization_sweep,
    simulate_photon_capture,
    sinusoid_pattern,
    unsharp_mask,
)

plt.rcParams["figure.figsize"] = (6, 4)

## 1. Local Filtering: What Does a Kernel Do?

A linear filter replaces each pixel by a weighted combination of nearby pixels. Move the entries of the kernel and compare the filtered result to the input image.

Exploration missions:

- What changes when the kernel sum is positive, negative, or zero?
- What changes when the kernel sum has large magnitude?
- What do negative output values mean, and where do they appear?
- Can you find a filter that enhances vertical edges?
- Can you find a filter that enhances horizontal edges?
- Can you find a filter that blurs the image?


In [ ]:
img = load_image("polar-bear.jpg")
kernel_state = kernel_playground(img)

After moving the sliders, run the next cell to inspect the latest kernel and filtered image returned by the widget. Compare the numbers to what you see in the plot. What changed relative to the input image?


In [ ]:
filtered = kernel_state["filtered"]
kernel = kernel_state["kernel"]
input_image = kernel_state["input"]

print("kernel:\n", kernel)
print("kernel sum:", kernel.sum())
print("input average:", input_image.mean())
print("filtered average:", filtered.mean())

## 2. Blurring Kernels

Blur removes detail by averaging nearby pixels, but different averages behave differently. Compare a box filter and a Gaussian filter. Inspect the kernel, the blurred image, and the residual.

Things to notice and explore:

- Play with different sizes and sigmas. Can you find Gaussian settings that behave almost like a box filter? What relationship between `size_x`, `size_y`, `sigma_x`, and `sigma_y` seems to make this happen?
- Can you find Gaussian settings where increasing the kernel size barely changes the output? What does this suggest about how much of the Gaussian is actually being used?
- Can you make the blur strong in one direction but weak in the other? What image structures are removed, and what structures remain?
- Can you create residuals that look like edge detectors in the horizontal direction, vertical direction, or both?


In [ ]:
img = load_image("polar-bear.jpg")
blur_controls = blur_comparison_widget(img)

### Covariance and Directional Blur

In the previous activity, `sigma_x` and `sigma_y` controlled the spread of an axis-aligned Gaussian (or a diagonal covariance matrix):

$$
\Sigma =
\begin{bmatrix}
\sigma_x^2 & 0\\
0 & \sigma_y^2
\end{bmatrix}.
$$

A general Gaussian with covariance matrix $\Sigma$ can be written as

$$
G(p)=\exp\left(-\frac{1}{2}p^T\Sigma^{-1}p\right),
\qquad
p=\begin{bmatrix}x\\y\end{bmatrix}.
$$

What changes if the covariance matrix is not diagonal?

In [ ]:
matrix_controls = matrix_gaussian_explorer()

A full $2\times2$ covariance matrix has four entries:

$$
\Sigma =
\begin{bmatrix}
a & b\\
c & d
\end{bmatrix}.
$$

A convenient way to define a valid covariance matrix is to choose two spreads and a rotation:

$$
\Sigma = R(\theta)
\begin{bmatrix}
\sigma_1^2 & 0\\
0 & \sigma_2^2
\end{bmatrix}
R(\theta)^T,
\qquad
R(\theta)=
\begin{bmatrix}
\cos\theta & -\sin\theta\\
\sin\theta & \cos\theta
\end{bmatrix}.
$$

The diagonal covariance matrix from the previous activity is the special case $\theta=0$. Then $R(\theta)$ is the identity matrix, so the rotation matrices do not change the axes.

In this construction, the eigenvectors are the columns of $R(\theta)$ and the eigenvalues are $\sigma_1^2$ and $\sigma_2^2$.

Before using this for rotated filtering, explore what eigenvectors and eigenvalues do.

An eigenvector $v$ satisfies

$$
\Sigma v = \lambda v.
$$

Multiplying by $\Sigma$ changes its length but not its direction. Use the sliders below to rotate the covariance matrix and move the test vector. Find vectors whose direction stays fixed after multiplication by $\Sigma$.

Questions to investigate:

- Which directions are the eigenvectors?
- What happens when the test vector is not an eigenvector?
- How are the eigenvalues related to `sigma_1` and `sigma_2`?
- What changes when `sigma_1 = sigma_2`?


In [ ]:
covariance_controls = covariance_explorer()

### Rotated Gaussian Challenge

Start with the diagonal Gaussian below. Then write a new function that uses a rotation matrix and the covariance equation from above.

In [ ]:
def make_gaussian_filter_diagonal(size, sigma_1, sigma_2):
    if size % 2 == 0:
        size += 1
    radius = size // 2
    y, x = np.mgrid[-radius:radius + 1, -radius:radius + 1]

    values = np.exp(-(x**2 / (2 * sigma_1**2) + y**2 / (2 * sigma_2**2)))
    kernel = values / values.sum()
    covariance = np.array([
        [sigma_1**2, 0],
        [0, sigma_2**2],
    ])
    return kernel, covariance

img = load_image("polar-bear.jpg")
small_img = resize_image(rgb_to_gray(img), output_shape=(220, 330))
kernel, covariance = make_gaussian_filter_diagonal(
    size=21,
    sigma_1=5,
    sigma_2=1.6,
)
filtered = convolve_image(small_img, kernel)
eigenvalues, eigenvectors = np.linalg.eigh(covariance)

print("covariance:\n", covariance)
print("eigenvalues:", eigenvalues)
print("eigenvectors:\n", eigenvectors)

fig, ax = plt.subplots(1, 3, figsize=(11, 3.5))
show(small_img, "Input", ax[0])
ax[1].imshow(kernel, cmap="viridis")
ax[1].set_title("Diagonal Gaussian kernel")
ax[1].set_axis_off()
show(np.clip(filtered, 0, 1), "Convolved image", ax[2])
plt.tight_layout()

## 3. Sharpening and Halos

In the blurring activity, the residual looked like an edge image: it kept what was removed by the blur. We can make this decomposition explicit. First, blur the image with a blur kernel $k_{\text{blur}}$:

$$
I_{\text{blur}} = I * k_{\text{blur}}.
$$

The detail image is the information removed by that blur:

$$
I_{\text{detail}} = I - I_{\text{blur}}.
$$

Sharpening adds a chosen amount $\alpha$ of this detail back to the original image:

$$
I_{\text{sharpened}} = I + \alpha I_{\text{detail}}.
$$

Because

$$
I = I_{\text{blur}} + I_{\text{detail}},
$$

the same expression can be written as

$$
I_{\text{sharpened}}
= I_{\text{blur}} + (1+\alpha)I_{\text{detail}},
$$

or, after substituting the definition of the detail image,

$$
I_{\text{sharpened}}
= I + \alpha\left(I-I*k_{\text{blur}}\right).
$$

In the widget:

- `radius` is the Gaussian standard deviation used to construct $k_{\text{blur}}$. It therefore controls the scale at which the image is separated into blur and detail.
- `amount` is $\alpha$. It controls how much additional detail is added to the original image.

This can make boundaries look clearer, but it can also amplify noise and create halos.

Exploration missions:

- Before moving the controls, predict what increasing `amount` will do while `radius` remains fixed.
- Increase `amount` slowly. At what point do edges stop looking sharper and start looking outlined?
- Before changing `radius`, predict which structures will appear in $I_{\text{detail}}$ for small and large values.
- Compare sharpening with a small `radius` versus a large `radius`. Which one changes fine texture, and which one changes larger object boundaries?
- Can the same parameter choice improve one region while damaging another? Find and explain such a tradeoff.
- Experiment with your own images.

In [ ]:
img = load_image("kings-college-circle.jpg")
sharp_controls = sharpening_widget(img)

## 4. Bilateral Filtering

A Gaussian filter averages nearby pixels, with closer pixels receiving more weight. A bilateral filter adds a second condition: nearby pixels receive substantial weight only when their intensity or color is also similar to that of the center pixel.

For a center pixel $p$ and one of its neighbours $q$, define the bilateral weight $
w(p,q)
=
G_s(p,q)\,G_r(p,q),
$

where

$$
G_s(p,q)
=
\exp\left(
-\frac{\|p-q\|^2}{2\sigma_s^2}
\right)
$$

measures spatial closeness, and

$$
G_r(p,q)
=
\exp\left(
-\frac{\|I(p)-I(q)\|^2}{2\sigma_r^2}
\right)
$$

measures similarity in intensity or color.

In words,

$$
\text{weight}
=
\text{spatial closeness}
\times
\text{appearance similarity}.
$$

The output at $p$ is a normalized weighted average of the pixels in its neighbourhood:

$$
I_{\text{out}}(p)
=
\frac{
\displaystyle\sum_{q\in N(p)}w(p,q)I(q)
}{
\displaystyle\sum_{q\in N(p)}w(p,q)
}.
$$

We give the denominator its own name:

$$
W_p
=
\sum_{q\in N(p)}w(p,q).
$$

Therefore, the same output equation can be written as

$$
I_{\text{out}}(p)
=
\frac{1}{W_p}
\sum_{q\in N(p)}w(p,q)I(q).
$$

Here:

- $p$ is the center pixel currently being filtered.
- $q$ is one candidate neighbouring pixel.
- $N(p)$ is the neighbourhood examined around $p$.
- $I(p)$ and $I(q)$ are the pixel intensities or color vectors at $p$ and $q$.
- $\|p-q\|$ is the spatial distance between the pixels.
- $\|I(p)-I(q)\|$ is their difference in intensity or color.
- $\sigma_s$ controls the spatial scale used by $G_s$.
- $\sigma_r$ controls the intensity or color scale used by $G_r$.
- $W_p$ is the sum of all weights. Dividing by it makes the normalized weights sum to one, so the result is a weighted average rather than an arbitrarily scaled sum.

For grayscale images, $\|I(p)-I(q)\|$ is simply the absolute intensity difference $|I(p)-I(q)|$. For color images, it is the distance between the two color vectors.

Unlike an ordinary Gaussian filter, the bilateral weights depend on the image values themselves. Two pixels at the same spatial distance from $p$ can therefore receive very different weights.

Click a center pixel in the diagonal-edge image. The red circle shows the one-standard-deviation spatial scale $\sigma_s$; it does not mark a hard cutoff. Pixels outside the circle may still contribute with smaller spatial weights. Compare the range weights, the weighted neighbourhood values, the Gaussian result, and the bilateral result.

Exploration missions:

- Before clicking, predict how the weights will differ for a center pixel near the edge and one far from the edge.
- Click pixels near and far from the diagonal edge. Which nearby pixels receive large weights, and which receive small weights?
- Increase the noise. Find values of $\sigma_s$ and $\sigma_r$ that remove noise without noticeably diluting the edge.
- Predict what will happen to the range weights as $\sigma_r$ increases, then test your prediction.
- Increase $\sigma_r$ until the bilateral result resembles the Gaussian result. What changed in the range weights?

In [ ]:
%matplotlib widget
bilateral_edge_state = bilateral_edge_widget()

## 5. Fourier Transform

Start with a 1D signal. Increase the number of Fourier components and watch the reconstruction change.

Exploration missions:

- What does `components = 0` reconstruct?
- What does the first component add?
- Which parts of the signal appear with only a few components?
- Where does the residual remain largest?
- What happens near a sharp jump?


In [ ]:
fourier_1d_controls = fourier_1d_reconstruction_widget()

### 2D Fourier Components

The Fourier transform of a 2D image represents the image as a sum of sinusoidal image patterns. Each point in the Fourier spectrum corresponds to one frequency direction and spacing.

Move one Fourier component around the spectrum. The selected point and its conjugate pair define a 2D sinusoidal pattern.

Exploration missions:

- How does distance from the center change stripe spacing?
- How does direction from the center change stripe orientation?
- Which selected components correspond to broad shading?
- Which selected components correspond to fine texture?


In [ ]:
malinois = load_image("malinois-puppies.jpg")
orchestra = load_image("orchestra.jpg")
component_controls = fourier_component_widget(malinois, "Malinois")

Now keep all Fourier components within a radius around the center. Increasing the radius adds higher spatial frequencies.

Exploration missions:

- Compare the Fourier magnitudes of the Malinois and orchestra images. Which spectrum has stronger horizontal and vertical structure? Which one looks more evenly spread across directions?
- Look back at the images. What visual structures might explain those directions in the Fourier magnitude?
- Move the radius slider outward starting from 0. Which image becomes recognizable with fewer high frequencies?
- Near the outer parts of the spectrum, which image has more visible energy? What fine details in the image might be responsible?


In [ ]:
radius_controls = fourier_radius_reconstruction_widget(malinois, orchestra, "Malinois", "Orchestra")

Finally, separate magnitude from phase. Swap the magnitude and phase between the two images.

Exploration missions:

- Which swapped image is more recognizable?
- Does the recognizable structure seem to follow the magnitude or the phase?
- What image information seems to live in the magnitude spectrum?
- What image information seems to live in the phase spectrum?


In [ ]:
phase_swap_state = fourier_phase_swap_demo(malinois, orchestra, "Malinois", "Orchestra")

## 6. Hybrid Images

Hybrid images combine low frequencies from one image with high frequencies from another. What you perceive depends on scale and viewing distance.

$$
I_{low} = G_{\sigma_{low}} * I_1
$$

$$
I_{high} = I_2 - G_{\sigma_{high}} * I_2
$$

$$
I_{hybrid} = I_{low} + \lambda I_{high}
$$


Exploration missions:

- Which image dominates when viewed large?
- Which image dominates when viewed small or from far away?
- Move `view scale` from large to small. Where does your perception switch from one image to the other? How does this relate to the selected `high_sigma` \ `low_sigma`
- If `high_sigma` becomes larger, does the high-frequency image retain more or less high-frequency structure? Why?
- Use `tx` and `ty` to align the second image. How sensitive is the hybrid to alignment?
- Experiment with different images. What makes a pair of images work?

In [ ]:
low_img = load_image("aviad.jpg")
high_img = load_image("len.jpg")
hybrid_controls = hybrid_image_widget(low_img, high_img)

The hidden target parameters below are not special. They just give you one fixed setting to compare against after you have played with the sliders.


In [ ]:
hybrid, low_part, high_part = make_hybrid_image(low_img, high_img, **HIDDEN_HYBRID_TARGET_PARAMS)

fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))
show(hybrid, "Hybrid image", ax[0])
show(resize_image(hybrid, scale=0.25), "Small / far view", ax[1])
plt.tight_layout()


## 7. Sampling and Aliasing

Sampling can turn high-frequency patterns into false low-frequency patterns. The image may show structure that was not really present in the original signal.

Exploration missions:

- Can you make a high-frequency pattern look like a low-frequency pattern?
- Can you make stripes appear to move in a different direction to the original pattern as you change frequency?
- Can you make a checkerboard turn into larger blocks or stripes after sampling?
- Which combinations of `freq` and `step` produce the strongest false pattern?
- Can you predict the sampled pattern before looking at the enlarged result?

In [ ]:
sampling_controls = sampling_widget()

## 8. Anti-aliasing by Blurring Real Images

Before reducing image resolution, we often blur the image. The blur removes frequencies that the smaller image cannot represent.

Exploration missions:

- Which textures alias most strongly?
- Can you find a blur strength that reduces aliasing but keeps the image useful?
- Why can the anti-aliased result look blurrier but more correct?


In [ ]:
downsample_controls = downsampling_widget(load_image("taormina.jpg"))

## 9. Final Mini-Challenge: Design a Filtering Pipeline

Build a small filtering pipeline for one goal: denoise, sharpen, downsample, remove repeated texture, or create a hybrid image. You may use any functions from this notebook.

Your answer should explain:

- What was your goal?
- Which parameters mattered most?
- What artifact did you create accidentally?
- What failure case did you find?
- How would you test whether your result is actually better?


In [ ]:
my_image = load_image("yonge-and-dundas.jpg")

# Example pipeline. Replace this with your own choices.
smoothed = gaussian_blur(my_image, sigma=1.0)
sharpened, blurred, detail = unsharp_mask(smoothed, radius=2.0, amount=1.0)
result = anti_alias_downsample(sharpened, step=2, sigma=1.0)

fig, ax = plt.subplots(1, 3, figsize=(11, 3.8))
show(my_image, "Input", ax[0])
show(sharpened, "Intermediate", ax[1])
show(result, "Result", ax[2])
plt.tight_layout()


# Homework 2: Coded Exposure and Motion Deblurring

Homework 2 is worth **5.00 marks**. Complete the six tasks below. The earlier lab explorations prepare you for the homework; they are not part of the marked submission. Keep your written responses beside their questions, and include your test code and implementations in the supplied code cells.

| Task | Marks |
|---|---:|
| Task 1: Design an exposure pattern | 0.25 |
| Task 2: Convert a PSF to a frequency response | 1.75 |
| Task 3: Interpret frequency responses | 0.75 |
| Task 4: Simulate motion blur and photon capture | 0.50 |
| Task 5: Investigate naive inversion | 0.75 |
| Task 6: Explore regularized inversion | 1.00 |
| **Total** | **5.00** |

Suppose a camera moves horizontally during an exposure. A conventional exposure records light throughout the motion, producing an approximately uniform motion-blur kernel. A coded exposure records light only at selected times, producing a less uniform blur kernel. The coded image can look unusual, but its kernel may preserve frequencies that conventional motion blur nearly removes.

If the sharp image is $X$, the blur kernel is $k$, and the observed image is $Y$, then

$$
Y = X * k.
$$

In the Fourier domain, convolution becomes multiplication:

$$
\widehat{Y} = H\widehat{X},
$$

where $H$ is the Fourier transform of the blur kernel. We will build this forward model one step at a time before trying to reverse it.

This simplified simulation assumes perfectly horizontal motion (therefore perfectly known blur kernels) and circular image boundaries. Note that: the coded kernel has lower total throughput, since we're capturing less light over the same duration.

## Task 1: Design an exposure pattern (0.25 marks)

Each entry in a horizontal motion kernel represents how much light was collected at one camera position. Every open time bin receives the same weight. A conventional exposure is open for all 16 bins and therefore has throughput one. A coded exposure may close any subset of the bins, so its **open fraction** is the number of open bins divided by 16. In this simplified model, the kernel's sum is the same fraction and measures its total light throughput.

The number of open bins is a design choice, so it is up to you to decide. More open bins collect more photons, but their arrangement may create unwanted frequency response patterns (as you will explore later). Fewer or more irregularly placed openings may improve recoverability while producing a darker, noisier observation. The goal is to investigate that tradeoff rather than assume one pattern is universally best.

The figure below shows both the **cause** and **spatial effect** for a conventional exposure and three example patterns. Each top-row stem plot is the blur kernel, also called the **point-spread function (PSF)**. It shows how much light is collected at each horizontal motion position. 

As you can see, a continuous uniform exposure makes a continuous streak; switching the exposure bins on and off produces a broken, nonuniform streak.


In [ ]:
coded_source = resize_image(
    rgb_to_gray(load_image("taito.jpg")),
    output_shape=(256, 256),
)

example_pattern_A = parse_exposure_pattern("1111111100000000")
example_pattern_B = parse_exposure_pattern("1010101010101010")
example_pattern_C = parse_exposure_pattern("1001011100101100")
example_patterns = {
    "A": example_pattern_A,
    "B": example_pattern_B,
    "C": example_pattern_C,
}

num_motion_positions = 16

uniform_motion_kernel = np.ones((1, num_motion_positions)) / num_motion_positions
example_motion_kernels = {
    name: make_exposure_kernel(pattern)
    for name, pattern in example_patterns.items()
}

point_image = make_gaussian_point_image(
    shape=(33, 64), center=(16, 32), sigma=1.2
)
schedule_examples = {
    "Conventional": uniform_motion_kernel,
    **{
        f"Pattern {name}": kernel
        for name, kernel in example_motion_kernels.items()
    },
}
point_response_examples = show_exposure_schedule_point_responses(
    point_image, schedule_examples
)
uniform_point_response = point_response_examples["Conventional"]
response_vmax = max(
    response.max() for response in point_response_examples.values()
)

### Design your own exposure pattern

Patterns A, B, and C all contain eight open bins, so their initial comparison holds photon throughput fixed and isolates the effect of *where* the openings occur.

### 1.0 Prediction [0 marks]

Before seeing the spectra, attempt to rank A, B, and C from easiest to hardest to reverse (de-blur). Explain your reasoning. Your predictions are not marked for correctness.

**1.0 Answer:** *Replace this text.*

### 1.1 Implementation [0.25 marks]

Edit `student_pattern_string` to contain 16 binary entries and at least one open bin. The number of ones is your choice. Run the cell and retain your final pattern, its open-bin count, throughput, and point-response plot.

Experiment with both the number and arrangement of open bins. Return to revise your design as you inspect the response curves and reconstructions. After revising it, rerun the dependent cells below so your final outputs use the same pattern.


In [ ]:
# 1.1 Implementation [0.25 marks]
# [Done]: replace this starter with your own 16-bin binary exposure pattern.
# student_pattern_string = "0000000100000000"
student_pattern_string = "1101001011100101"

student_pattern_string = student_pattern_string.strip()
assert len(student_pattern_string) == num_motion_positions, (
    f"Expected {num_motion_positions} bits."
)
assert set(student_pattern_string) <= {"0", "1"}, (
    "Use only 0 (closed) and 1 (open)."
)
assert "1" in student_pattern_string, "At least one time bin must be open."

student_exposure_pattern = parse_exposure_pattern(student_pattern_string)
coded_exposure_pattern = student_exposure_pattern
coded_motion_kernel = make_exposure_kernel(coded_exposure_pattern)

num_open_bins = int(coded_exposure_pattern.sum())
coded_duty_cycle = coded_motion_kernel.sum()
print(f"Open bins: {num_open_bins}/{num_motion_positions}")
print(f"Photon throughput H(0): {coded_duty_cycle:.3f}")

student_point_responses = show_exposure_schedule_point_responses(
    point_image,
    {f"Your design: {student_pattern_string}": coded_motion_kernel},
    response_vmax=response_vmax,
)
coded_point_response = next(iter(student_point_responses.values()))


## Task 2: Convert a PSF to a frequency response (1.75 marks)

In Task 1, each top-row stem plot showed a blur kernel. That kernel is the **point-spread function (PSF)**, usually written $k$. The captured-point image directly below it showed the same PSF spatially: if the input is one point, the PSF tells us where the point's light appears in the captured image.

The **optical transfer function (OTF)**, written $H$, describes the *same blur*, but in the frequency domain. It tells us what the blur does to sinusoidal image patterns at different frequencies. The two representations are related by a Fourier transform:

$$
H = \mathcal{F}\{k\}.
$$

This representation is useful because spatial convolution becomes pointwise multiplication:

$$
Y=X*k
\qquad\Longleftrightarrow\qquad
\widehat{Y}=H\widehat{X}.
$$

Before taking the FFT, two bookkeeping steps are required. First, $H$ must have the same shape as the image so that it can multiply $\widehat{X}$ element by element. Second, NumPy's FFT treats array index $(0,0)$ as the spatial origin, so the PSF's center must be moved there. You will need to implement the function that converts a PSF to an OTF. Conceptually, `psf_to_otf` does:

$$
\text{small PSF}
\rightarrow \text{pad to image size}
\rightarrow \text{move center to }(0,0)
\rightarrow \operatorname{fft2}
\rightarrow \text{OTF }H.
$$

### 2.1 Implementation [0.25 marks]

Implement `psf_to_otf` using the steps below. Your function must correctly pad, align, and transform arbitrary valid 2D kernels.


1. Convert `kernel` to a floating-point NumPy array.
2. Create a zero-filled array with shape `output_shape`.
3. Copy the `kernel` into the upper-left corner of that array.
4. Circularly shift it upward by `kernel_height // 2` and left by `kernel_width // 2`. This moves the kernel's center to the FFT origin. Use `np.roll` with axes `(0, 1)`.
5. Return `np.fft.fft2` of the shifted array.

The returned $H$ is complex-valued. Its magnitude $|H(f)|$ describes attenuation at frequency $f$, while its phase describes how that frequency is shifted. We will visualize the magnitude first.


In [ ]:
# 2.1 Implementation [0.25 marks]
def psf_to_otf(kernel, output_shape):
    """Convert a centered spatial PSF into an image-sized OTF.

    Parameters
    ----------
    kernel : array-like, shape (K_h, K_w)
        Real-valued 2D point-spread function. Its center is assumed to be
        at index `(K_h // 2, K_w // 2)`. Each kernel dimension must not
        exceed the corresponding output dimension.
    output_shape : tuple of int
        Requested `(height, width)` of the frequency response.

    Returns
    -------
    H : ndarray, shape `output_shape`, complex
        Optical transfer function aligned for multiplication with
        `np.fft.fft2` of an image having `output_shape`.
    """
    # [Done]: implement the five steps described above.
    # raise NotImplementedError

    # step1. Convert the kernel to a floating-point array.
    kernel = np.asarray(kernel, dtype=float)
    kernel_height, kernel_width = kernel.shape

    # step2. Create an image-sized zero array.
    padded = np.zeros(output_shape, dtype=float)

    # step3. Place the kernel in the upper-left corner.
    padded[:kernel_height, :kernel_width] = kernel

    # step4. Move the kernel's center to the FFT origin.
    shifted = np.roll(padded, shift=(-(kernel_height // 2), -(kernel_width // 2)), axis=(0, 1))

    # step5. Compute the frequency response.
    return np.fft.fft2(shifted)

**2.2 Testing [1.00 marks]**

Design and run one or more test cases. Construct an array for the PSF and an array for its expected OTF at the output array's discrete frequencies. Compare this expected OTF with the output of your `psf_to_otf`, and explain what your results tell you about its correctness.

Your answer should:

- Explain why this is a useful test and which implementation mistakes it could detect.
- Interpret the result.

Obtain the expected OTF independently of `psf_to_otf` and FFT calls. At least one case should have a nonconstant frequency response, so checking only an impulse or the output shape is not sufficient.

**2.2 Answer:**
- Test 1 uses an asymmetric 3×3 PSF padded to 4×4. Relative to its center, the nonzero weights are 0.5 at (0,0), 0.25 at
  (0,1), and 0.125 at (1,0). Thus, independently of FFT calls, H[u,v] = 0.5 + 0.25 exp(-2πiv/4) + 0.125 exp(-2πiu/4).
  This nonconstant complex response tests zero-padding, center alignment, shift direction, and row/column handling.

- Test 2 uses a 2×2 PSF with center (1,1) and a 3×4 output. The weight 0.25 lies one position left of the center, giving
  H[u,v] = 0.5 + 0.25 exp(2πiv/4), identical across rows. This tests the even-sized center convention and rectangular
  output handling.

- Both tests pass assert_allclose, showing that the computed complex OTFs agree with the independently derived expectations
  within numerical tolerance. This supports correctness for the tested cases, though it does not prove correctness for every PSF.

Replace the placeholder arrays in the next cell, name your test case, and add code to compare the computed OTF with your expected result. Add more test cases as needed.

In [ ]:
# 2.2 Testing [1.00 marks]
# Test case 1: Asymmetric odd-sized PSF with zero-padding.
test_psf = np.array([
    [0.0, 0.0,   0.0],
    [0.0, 0.5,   0.25],
    [0.0, 0.125, 0.0],
])

phase = np.array([1, -1j, -1, 1j], dtype = complex)
# H[u,v] = 0.5 + 0.25*exp(-2*pi*i*v/4) + 0.125*exp(-2*pi*i*u/4).
test_otf = (0.5 + 0.25 * phase[None, :] + 0.125 * phase[:, None])
computed_otf = psf_to_otf(test_psf, test_otf.shape)

assert computed_otf.shape == test_otf.shape
np.testing.assert_allclose(computed_otf, test_otf)
print("Test 1 passed.")
print("Maximum error:", np.max(np.abs(computed_otf - test_otf)))

# Test case 2: Even-sized PSF with a rectangular output.
test_psf = np.array([
    [0.0,  0.0],
    [0.25, 0.5],
])

# H[u,v] = 0.5 + 0.25*exp(2*pi*i*v/4).
test_otf = np.array([
    [0.75, 0.5 + 0.25j, 0.25, 0.5 - 0.25j],
    [0.75, 0.5 + 0.25j, 0.25, 0.5 - 0.25j],
    [0.75, 0.5 + 0.25j, 0.25, 0.5 - 0.25j],
], dtype=complex)
computed_otf = psf_to_otf(test_psf, test_otf.shape)

assert computed_otf.shape == test_otf.shape
np.testing.assert_allclose(computed_otf, test_otf)
print("Test 2 passed.")
print("Maximum error:", np.max(np.abs(computed_otf - test_otf)))

#### Visualize the output of `psf_to_otf`

`psf_to_otf` returns a complex-valued image-sized array. Each complex value contains a magnitude and a phase. Its zero-frequency value, $H(0)=\sum k$, measures total light throughput: it is $1.0$ for conventional exposure and equals the fraction of open bins for your coded design.

To compare *which frequencies* each kernel preserves independently of overall brightness, the plots show relative magnitude $|H(f)|/H(0)$. A relative value near one is preserved about as strongly as the average brightness, whereas a value near zero is almost erased. 

Note: For a general 2D blur, the OTF is also 2D because an image can vary horizontally and vertically. But because our simulated motion is purely horizontal, the response is the same in the vertical-frequency direction. 


In [ ]:
H_uniform = psf_to_otf(uniform_motion_kernel, coded_source.shape)
H_coded = psf_to_otf(coded_motion_kernel, coded_source.shape)
frequencies = np.fft.fftshift(np.fft.fftfreq(coded_source.shape[1]))

_ = show_frequency_responses(
    {
        "Conventional exposure": H_uniform,
        "Your coded exposure": H_coded,
    },
    frequencies,
)

### Compare exposure patterns in the frequency domain

The plots below display the relative magnitude $|H(f)|/H(0)$ over frequency. Note the locations and depths of the visible dips.

> **2.3 Question [0.25 marks]**
>
> Does distributing the open bins evenly always improve the frequency response? Compare patterns with the same number of open bins so that you isolate their arrangement. Refer to the response curves and explain what your comparison shows.

**2.3 Answer:** *No. A, B, and C each have eight open bins and throughput 0.5,
so the comparison isolates their arrangement. B has evenly spaced
openings, but its response has more zeros than A, including zeros
closer to zero frequency. C preserves many frequencies better,
although it still has deep dips and zeros. Thus, evenly distributing
open bins does not necessarily improve the frequency response.*

> **2.4 Question [0.25 marks]**
>
> Does adding more open bins necessarily improve invertibility (how simple it is to deblur)? Describe how you tested this, identify the patterns you compared, and use their response curves to explain your conclusion.

**2.4 Answer:** *No. I compared A (1111111100000000) with conventional exposure
(1111111111111111). Adding eight open bins increases throughput
from 0.5 to 1.0 but introduces additional response zeros. At
f = 1/16 cycles/pixel, the conventional response is zero, while
A's relative magnitude is approximately 0.64. Therefore, more
open bins collect more light but do not necessarily improve
invertibility, because they can cancel frequency information.*


In [ ]:
comparison_patterns = {
    **example_patterns,
    "Your design": student_exposure_pattern,
}
comparison_kernels = {
    name: make_exposure_kernel(pattern)
    for name, pattern in comparison_patterns.items()
}
comparison_otfs = {
    name: psf_to_otf(kernel, coded_source.shape)
    for name, kernel in comparison_kernels.items()
}

_ = show_frequency_responses(comparison_otfs, frequencies)

## Task 3: Interpret frequency responses (0.75 marks)

A horizontal sinusoidal pattern contains one horizontal frequency. After accounting for the output's average brightness, its relative stripe contrast is scaled by the normalized response:

$$
C_{\text{out}}(f_x)
=\frac{|H(f_x)|}{H(0)}\,C_{\text{in}}(f_x).
$$

Use the slider below to choose a sinusoid. The vertical marker locates that frequency on each response curve. All displayed images use the same intensity scale, so lower-throughput outputs remain visibly darker.

> **3.0 Prediction [0 marks]**

Before moving the slider, predict what will happen to the output stripes (a) as the selected frequency approaches a deep response dip and (b) at a frequency where the response is zero.

> **3.1 Question [0.25 marks]**
>
> Move the slider to a frequency where a response is near zero. State the pattern and selected frequency, describe what happens to the output stripe contrast, and explain why the original strength of that stripe pattern would be difficult to recover. Retain a screenshot showing the curve and filtered stripes.

**3.1 Answer:** *Selected conventional exposure at 16 cycles/image
(0.0625 cycles/pixel). The response curve reaches a zero,
and the filtered image becomes uniform grey with no visible
stripe contrast. The original contrast cannot be uniquely
recovered at an exact zero; near a zero, inversion amplifies noise.*

> **3.2 Question [0.50 marks]**
>
> At which nonzero frequency or frequencies do patterns A, B, and C all fail to retain the input detail? Use both the response curves and filtered stripe images as evidence. State the frequencies in cycles/image or cycles/pixel, label the units, and retain screenshots that support your answer.

**3.2 Answer:** *A, B, and C all lose the input detail at 64 cycles/image
(0.25 cycles/pixel). Their response curves have a common zero
at this frequency. The filtered images confirm this: all three
show relative responses of 0.000 and become uniform grey,
with no visible stripe contrast.*


In [ ]:
frequency_cases = {"Conventional": H_uniform, **comparison_otfs}

frequency_controls = coded_exposure_frequency_widget(
    coded_source, frequency_cases, value=8, max_cycles=128
)
frequency_control = frequency_controls["cycles_per_image"]


## Task 4: Simulate motion blur and photon capture (0.50 marks)

We will implement `circular_blur(image, kernel)` using the same model:

$$
\widehat{Y}=H\widehat{X}.
$$

### 4.1 Implementation [0.25 marks]

Implement the function so that it (1) computes $H$ with `psf_to_otf`, (2) transforms the image with `np.fft.fft2`, (3) multiplies by $H$, and (4) applies `np.fft.ifft2` and `np.real`. Return both the blurred image and $H$.

This function simulates the blurred image a camera would capture for each exposure pattern. The next experiment adds photon-counting noise to these images, then Tasks 5 and 6 use the observations and returned OTFs to attempt deblurring. Using the same known kernel for capture and reconstruction lets you study the effect of the exposure pattern and inverse method. Here, "circular" describes the boundary rule: content that moves beyond one edge wraps around to the opposite edge. It does not mean the camera moves in a circle.


In [ ]:
# 4.1 Implementation [0.25 marks]
def circular_blur(image, kernel):
    """Blur a grayscale image using circular-boundary convolution.

    Parameters
    ----------
    image : array-like, shape (H, W)
        Real-valued 2D grayscale image.
    kernel : array-like, shape (K_h, K_w)
        Real-valued centered PSF whose dimensions do not exceed the image.

    Returns
    -------
    blurred : ndarray, shape (H, W)
        Real-valued blurred image. The FFT model wraps image content across
        opposite boundaries.
    H : ndarray, shape (H, W), complex
        OTF used to create `blurred`.
    """
    # [Done]: implement the four steps described above.
    # raise NotImplementedError

    # step1: Compute the image-sized OTF.
    image = np.asarray(image, dtype=float)
    H = psf_to_otf(kernel, image.shape)

    # step2: Transform the image to the frequency domain.
    image_fft = np.fft.fft2(image)

    # step3: Multiply by the OTF.
    blurred_fft = H * image_fft

    # step4: Transform back and take the real part.
    blurred = np.real(np.fft.ifft2(blurred_fft))

    return blurred, H

In [ ]:
# Optional self-checks for circular_blur (0 marks).
# Use this cell if you want to check your implementation before continuing.

# Test 1: A right-offset impulse shifts the image right by one pixel.
image = np.array([[1., 2., 3., 4.], [5., 6., 7., 8.]])
kernel = np.array([[0., 0., 1.]])
expected = np.array([[4., 1., 2., 3.], [8., 5., 6., 7.]])
blurred, H = circular_blur(image, kernel)

assert H.shape == image.shape
assert np.iscomplexobj(H)
np.testing.assert_allclose(blurred, expected, atol=1e-12)
print("Test 1 passed.")

# Test 2: A constant image is scaled by the kernel sum.
image = np.full((4, 5), 0.8)
kernel = np.array([[0.25, 0.25]])  # Total throughput = 0.5.
blurred, H = circular_blur(image, kernel)

np.testing.assert_allclose(blurred, 0.4, atol=1e-12)
np.testing.assert_allclose(H[0, 0], 0.5, atol=1e-12)
assert np.isrealobj(blurred)
print("Test 2 passed.")

### Generate controlled observations

The next cell applies the motion blur from each exposure design to the same source image. It treats each blurred pixel value as an expected photon count and samples Poisson noise. Coded observations collect less light when fewer bins are open, and their relative uncertainty from photon noise is larger.

All images use the same display intensity scale. The printed means let you distinguish brightness changes from loss of detail.

> **4.2 Question [0.25 marks]**
>
> Compare the mean intensity of your coded observation with the sharp source and the conventional observation. Is the brightness change consistent with your open fraction, $H(0)$? Use the printed means to support your answer. If you rescaled the coded observation to match the source's mean brightness, would that also remove the additional relative photon noise? Explain why or why not.

**4.2 Answer:**
*The sharp source mean is 0.2574, the conventional observation mean is 0.2573, and the coded observation mean is 0.1448.
The coded-to-source ratio is approximately 0.5625, matching the open fraction H(0) = 0.5625. Conventional exposure preserves
the mean brightness, while the coded observation is darker.
Rescaling the coded image would restore its mean brightness but would not remove the additional relative photon noise,
because signal and noise are amplified equally. The greater relative uncertainty caused by collecting fewer photons remains.*


In [ ]:
capture_kernels = {"Conventional": uniform_motion_kernel, **comparison_kernels}
blurred_by_pattern = {}
H_by_pattern = {}
for name, kernel in capture_kernels.items():
    blurred_by_pattern[name], H_by_pattern[name] = circular_blur(
        coded_source, kernel
    )

# Same acquisition settings, but independent photon-counting noise.
rng = np.random.default_rng(320)
photons_per_unit_intensity = 2000

observed_by_pattern = {
    name: simulate_photon_capture(
        blurred, photons_per_unit_intensity, rng
    )
    for name, blurred in blurred_by_pattern.items()
}

# Convenient aliases used by later focused comparisons.
uniform_blurred = blurred_by_pattern["Conventional"]
coded_blurred = blurred_by_pattern["Your design"]
H_uniform = H_by_pattern["Conventional"]
H_coded = H_by_pattern["Your design"]
uniform_observed = observed_by_pattern["Conventional"]
coded_observed = observed_by_pattern["Your design"]

capture_titles = {
    name: f"{name} blur\nthroughput={kernel.sum():.3f}"
    for name, kernel in capture_kernels.items()
}
_ = show_image_case_grid(
    coded_source, observed_by_pattern, titles=capture_titles
)

print(f"Sharp source mean: {coded_source.mean():.4f}")
print(f"Conventional observation mean: {uniform_observed.mean():.4f}")
print(f"Your coded observation mean: {coded_observed.mean():.4f}")
print(f"Your open fraction H(0): {float(np.real(H_coded[0, 0])):.4f}")

## Task 5: Investigate naive inversion (0.75 marks)

Because $\widehat{Y}=H\widehat{X}$, the most direct attempt at deblurring is

$$
\widehat{X}_{\text{naive}}(f)=\frac{\widehat{Y}(f)}{H(f)}.
$$

### 5.1 Implementation [0.25 marks]

Implement `naive_inverse_filter` by transforming `observed`, dividing by `H + epsilon`, applying the inverse FFT, and returning the real part.

The next comparison applies the same naive inverse to Conventional, A, B, C, and Your design. A pattern whose *relative* response stays away from zero may retain more recognizable structure. However, dividing by a lower-throughput $H$ also restores brightness by amplifying photon noise. A high-throughput pattern can still fail if it creates deep nulls. This is why spectral recoverability and photon efficiency must be considered together.


In [ ]:
# 5.1 Implementation [0.25 marks]
def naive_inverse_filter(observed, H, epsilon=1e-6):
    """Deblur an observation by direct frequency-domain division.

    Parameters
    ----------
    observed : array-like, shape (H, W)
        Real-valued blurred grayscale image.
    H : array-like, shape (H, W), complex
        Known OTF that produced `observed`; it must have the same shape.
    epsilon : float, optional
        Small positive value added to the denominator to avoid literal
        division by zero. It does not prevent noise amplification.

    Returns
    -------
    reconstruction : ndarray, shape (H, W)
        Real-valued, unclipped estimate of the sharp image. Values may lie
        outside `[0, 1]`.
    """
    # [Done]: implement the direct frequency-domain inverse.
    # raise NotImplementedError
    observed_fft = np.fft.fft2(observed)
    reconstruction_fft = observed_fft / (H + epsilon)
    reconstruction = np.real(np.fft.ifft2(reconstruction_fft))
    return reconstruction

#### Inspect the naive reconstruction

The images are clipped for display, so also inspect the printed un-clipped value ranges. Extremely large positive and negative values are evidence of noise amplification even if clipping hides some of it.

> **5.2 Question [0.50 marks]**
>
> Using the reconstruction images, the printed unclipped value ranges, and the earlier response curves, explain when and why the naive inverse fails. Distinguish an exactly zero response from a small nonzero response in the presence of noise. Does adding `epsilon` solve both problems? Support your explanation with at least one named exposure pattern from your results.

**5.2 Answer:**
- At H(f) = 0, the corresponding image information is lost and
  cannot be uniquely recovered. For small nonzero H(f), information
  remains, but inversion strongly amplifies noise by about 1/|H(f)|.
- Conventional exposure has many response zeros. Its reconstruction
  is dominated by black-and-white artifacts, with an unclipped range
  of [-10086.508, 11161.211], showing severe noise amplification.
- My design has no visible response zeros and retains recognizable
  buildings, but its deep dips still amplify noise. Its unclipped
  range is [-0.544, 1.455].
- Adding epsilon prevents division by zero at H(f) = 0, but neither
  restores lost information nor adequately suppresses amplified noise.
  Display clipping only hides the extreme values.

In [ ]:
naive_by_pattern = {
    name: naive_inverse_filter(observed_by_pattern[name], H_by_pattern[name])
    for name in observed_by_pattern
}
uniform_naive = naive_by_pattern["Conventional"]
coded_naive = naive_by_pattern["Your design"]

naive_display = {
    name: np.clip(reconstruction, 0, 1)
    for name, reconstruction in naive_by_pattern.items()
}
naive_titles = {name: f"Naive inverse: {name}" for name in naive_by_pattern}
show_image_case_grid(coded_source, naive_display, titles=naive_titles)

for name, reconstruction in naive_by_pattern.items():
    print(
        f"{name:>12} inverse range: "
        f"[{reconstruction.min():.3f}, {reconstruction.max():.3f}]"
    )

## Task 6: Explore regularized inversion (1.00 marks)

The naive inverse treats every frequency as trustworthy. A regularized inverse deliberately stops trying to recover frequencies where the blur response is too weak. This introduces some bias (remaining blur) in exchange for much less noise amplification:

$$
\widehat{X}_{\text{estimate}}
= \widehat{Y}
\frac{H^*}{|H|^2 + \lambda},
$$

Here, $H^*$ is the complex conjugate of $H$. The user-chosen parameter $\lambda>0$ keeps the denominator away from zero. A smaller $\lambda$ tries to invert the blur more aggressively but amplifies more noise; a larger $\lambda$ suppresses noise but leaves more blur. This $\lambda$ is unrelated to the eigenvalues used earlier in the notebook.

### 6.1 Implementation [0.25 marks]

Implement the function from the formula.

The naive inverse is the aggressive limiting idea as regularization approaches zero, while larger values increasingly suppress uncertain frequencies.


In [ ]:
# 6.1 Implementation [0.25 marks]
def regularized_inverse_filter(observed, H, regularization):
    """Deblur an observation with a regularized frequency-domain inverse.

    Parameters
    ----------
    observed : array-like, shape (H, W)
        Real-valued blurred grayscale image.
    H : array-like, shape (H, W), complex
        Known OTF that produced `observed`; it must have the same shape.
    regularization : float
        Positive strength `lambda`. Larger values suppress uncertain
        frequencies more strongly but generally leave more blur.

    Returns
    -------
    reconstruction : ndarray, shape (H, W)
        Real-valued estimate clipped to the display range `[0, 1]`.
    """
    # [Done]: implement the regularized frequency-domain inverse.
    # raise NotImplementedError
    observed_fft = np.fft.fft2(observed)
    inverse_response = np.conj(H) / (np.abs(H)**2 + regularization)
    reconstruction = np.real(np.fft.ifft2(observed_fft * inverse_response))
    return np.clip(reconstruction, 0.0, 1.0)

#### Compare exposure patterns at the same regularization

The grid below uses $\lambda=10^{-2}$. 

In [ ]:
comparison_regularization = 1e-2
regularized_by_pattern = {
    name: regularized_inverse_filter(
        observed_by_pattern[name], H_by_pattern[name], comparison_regularization
    )
    for name in observed_by_pattern
}

regularized_titles = {
    name: (
        rf"{name}: $\lambda={comparison_regularization:.0e}$\n"
        f"MSE={np.mean((coded_source - reconstruction) ** 2):.4f}"
    )
    for name, reconstruction in regularized_by_pattern.items()
}
show_image_case_grid(
    coded_source,
    regularized_by_pattern,
    titles=regularized_titles,
    suptitle="Same regularization, different exposure patterns",
)

# Keep these results for the next comparison and final reflection.
uniform_reconstruction = regularized_by_pattern["Conventional"]
coded_reconstruction = regularized_by_pattern["Your design"]
uniform_mse = np.mean((coded_source - uniform_reconstruction) ** 2)
coded_mse = np.mean((coded_source - coded_reconstruction) ** 2)

### Changing the regularization factor

Keep each observation fixed and vary only $\lambda$ for Conventional exposure (top) and Your design (bottom). The next plot reports mean squared error (MSE) relative to the known sharp source; lower MSE means a closer pixelwise reconstruction. The values are also retained in `sweep_mses`.

> **6.2 Question [0.50 marks]**
>
> Discuss quantitatively and qualitatively how $\lambda$ affects Conventional exposure and your coded design. For each, compare at least two displayed $\lambda$ settings: quote their MSE values and describe the visible changes. Explain the tradeoff behind your observations and how you decided what is the best $\lambda$ for your pattern.

**6.2 Answer:**
- Conventional exposure: Increasing lambda from 1e-3 to 1e-2
  reduces MSE from 0.0157 to 0.0054 and visibly reduces noise.
  At 1e-1, the image becomes smoother but more blurred, and
  MSE increases to 0.0087.
- My coded design: Increasing lambda from 1e-3 to 1e-2 reduces
  MSE from 0.0060 to 0.0045, suppressing noise while retaining
  building structure. At 1e-1, fine window details are lost,
  the image becomes darker, and MSE rises to 0.0145.
- Larger lambda limits noise amplification but suppresses more
  image detail. I choose lambda = 1e-2 for my design because it
  has the lowest MSE among the displayed settings and provides
  a good visual balance between noise reduction and sharpness.

In [ ]:
regularization_values = [1e-5, 1e-3, 1e-2, 1e-1]
sweep_reconstructions, sweep_mses = show_regularization_sweep(
    coded_source,
    {
        "Conventional exposure": uniform_observed,
        "Your design": coded_observed,
    },
    {
        "Conventional exposure": H_uniform,
        "Your design": H_coded,
    },
    regularization_values,
    regularized_inverse_filter,
)

### System-design tradeoff

Some coded patterns can be easier to invert than conventional motion blur, yet other capture strategies may still be preferable in practice.

> **6.3 Question [0.25 marks]**
>
> State one reason a camera system might not use coded exposure for every photograph. Explain when that limitation would matter, connecting it to an assumption of this simulation or an effect you observed.

**6.3 Answer:** *Coded exposure collects less light, which is a disadvantage in low-light conditions where photon noise is already significant.
My pattern had a throughput of 56.25% relative to conventional exposure, reducing the mean intensity from 0.2574 to 0.1448.
Restoring brightness also amplifies noise, so improved invertibility may not compensate for the loss of photons.*

## Task summary and submission checklist (5.00 marks total)

| Task | Marks |
|---|---:|
| 1.1 Implementation | 0.25 |
| 2.1 Implementation | 0.25 |
| 2.2 Testing | 1.00 |
| 2.3 Question | 0.25 |
| 2.4 Question | 0.25 |
| 3.1 Question | 0.25 |
| 3.2 Question | 0.50 |
| 4.1 Implementation | 0.25 |
| 4.2 Question | 0.25 |
| 5.1 Implementation | 0.25 |
| 5.2 Question | 0.50 |
| 6.1 Implementation | 0.25 |
| 6.2 Question | 0.50 |
| 6.3 Question | 0.25 |
| **Total** | **5.00** |

### Before submission

- Complete your exposure design and all four required functions: `psf_to_otf`, `circular_blur`, `naive_inverse_filter`, and `regularized_inverse_filter`.
- Run and retain your 2.2 tests, including the expected results, actual results, and explanations.
- Replace every required written-answer placeholder beside its question. Optional predictions are not graded.
- Retain the plots and numerical outputs that support your answers. If you revise your exposure pattern, rerun the dependent cells so the final results are consistent.
- Save screenshots of the frequency-probe widget at the settings used in your Task 3 answers. A saved notebook may not retain every previous slider setting.
- Keep supplied function signatures, photon settings, random seed, and metric definitions unchanged. Edit the designated implementations, pattern comparisons, and experiment settings as instructed.
- Submit the completed notebook and the PDF report described below.


# Submission instructions

For Homework 2, please submit **exactly two files**:

1. **Your completed Jupyter notebook (`.ipynb`)**
2. **One PDF report (`.pdf`)**

You do **not** need to submit the supplied images, helper modules, or original lab ZIP separately. Include the relevant output images and plots in the PDF as described below. This homework uses the supplied image; you do not need to capture your own photographs.

## 1. Completed Jupyter notebook

Your notebook must include:

- All completed code and written answers for Homework 2
- Every required homework `TODO` replaced
- The original supplied cells and function signatures, with your work filled into the designated locations
- Saved outputs from the relevant cells, including test results, plots, and numerical results

Before submitting, run the required cells and save the notebook. When restarting the kernel between interactive sections, do **not** clear the outputs that you need to submit. After changing your exposure pattern, rerun the dependent cells so your submitted outputs reflect your final design.

### Confirm that:

- Your final exposure pattern, open-bin count, throughput, and point-response plot are visible
- Your four function implementations and your 2.2 test code and results are present
- The frequency-response plots, captured observations, naive reconstructions, and regularized reconstructions are visible
- The naive reconstruction ranges and regularization-sweep MSE values are retained
- Your written answers are complete and the PDF includes the frequency-probe screenshots used as evidence

## 2. PDF report

Organize the PDF by task and label each figure with its exposure pattern and relevant settings. Screenshots must be large enough for us to read the plot axes, curve legends, selected frequency, parameter values, and numerical results. You may export figures directly instead of taking screenshots if all relevant information is retained. Use multiple pages if needed to keep wide plots readable.

### Task 1 PDF contents

- Your final 16-bin exposure pattern
- The exposure and point-response plot for your design

### Task 2 PDF contents

- Snippet of your implemented function: `psf_to_otf`
- Your answer for 2.2, including any of the PSF definition and analytical OTF formula that you used, and your reasoning on why you chose the test case(s).
- The frequency-response comparison plots used in your answers, with the compared patterns identified
- Your written answers for 2.3 and 2.4

### Task 3 PDF contents

- Screenshots of the frequency-probe widget at the settings used for 3.1 and 3.2, showing the input stripes, filtered stripes, response curves, and selected frequency
- Your written answers for 3.1 and 3.2, including frequency values and units

### Task 4 PDF contents

- Snippet of your implemented function: `circular_blur`
- The capture comparison grid showing the sharp source and noisy blurred observations
- The printed mean intensities and your coded throughput
- Your written answer for 4.2

### Task 5 PDF contents

- Snippet of your implemented function: `naive_inverse_filter`
- The naive reconstruction comparison grid
- The printed unclipped reconstruction ranges
- Your written answer for 5.2

### Task 6 PDF contents

- Snippet of your implemented function: `regularized_inverse_filter`
- The reconstruction comparison grid at the same regularization strength
- The regularization grid plot for Conventional exposure and your design, with $\lambda$ and MSE values readable
- Your written answers for 6.2 and 6.3

Include your written explanations in both the notebook and PDF.
